# 🐷 Pig Valence Classification — SampleCNN on Raw Audio

**Goal:** Binary classification of pig vocalisations (**Positive** vs **Negative** valence)  
from **raw audio waveforms** using a PyTorch re-implementation of **SampleCNN**.

> Lee et al. (2017) *Sample-level Deep Convolutional Neural Networks  
> for Music Auto-tagging Using Raw Waveforms* — arXiv:1703.01789

---

### Why raw waveform instead of a spectrogram?

| Approach | Input | Feature extraction |
|---|---|---|
| Spectrogram CNN | Log-Mel image | **Hand-crafted** (STFT → Mel → log) |
| **SampleCNN** | Raw samples | **Learned** by the first strided convolution |

SampleCNN lets the first convolution layer learn its own filterbank directly  
from data — potentially discovering spectral patterns that a fixed Mel  
filterbank would miss for non-musical sounds like pig vocalisations.

---

### Pipeline
```
annotations.xlsx + .wav files
        │
        ▼
Group-aware split  (by Recording Team → prevents pig/site leakage)
        │
        ▼
PigAudioDataset  ──load .wav──►  pad/truncate to 16 384 samples
        │
        ▼
SampleCNN
  Conv0 (stride 2)  — learned filterbank
  13 × [Conv → BN → ReLU → MaxPool(2)]  — feature extraction
  Conv 1×1  — channel mixing
  Dropout → Linear → logits
        │
        ▼
Training loop  ──►  best_model.pth
        │
        ▼
Evaluation: accuracy · AUC-ROC · confusion matrix
```

---
## 1 · Install & import dependencies

In [ ]:
# Run once (or on every Colab session — packages reset between sessions)
!pip install -q torch torchvision torchaudio librosa pandas openpyxl \
               scikit-learn matplotlib seaborn soundfile

In [ ]:
import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
)

try:
    import librosa
    LIBROSA_AVAILABLE = True
except ImportError:
    import soundfile as sf
    LIBROSA_AVAILABLE = False
    print('librosa not found — using soundfile (no resampling support)')

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

---
## 2 · Configuration

**Edit the paths below** before running anything else.

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
ANNOTATIONS_FILE = 'annotations.xlsx'   # path to annotation spreadsheet
AUDIO_DIR        = 'audio/'             # folder containing the .wav files
BEST_MODEL_PATH  = 'best_samplecnn.pth'

# ── Google Colab: mount Drive if needed ───────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# ANNOTATIONS_FILE = '/content/drive/MyDrive/YOUR_FOLDER/annotations.xlsx'
# AUDIO_DIR        = '/content/drive/MyDrive/YOUR_FOLDER/audio/'

# ── Audio ──────────────────────────────────────────────────────────────────
SAMPLE_RATE   = 44_100
# SampleCNN requires a fixed input length.
# We choose 16 384 = 2^14 samples ≈ 0.37 s at 44 100 Hz.
# This powers exactly 1 strided-conv (÷2) + 13 MaxPool(÷2) layers = 2^14 total
# downsampling, collapsing the temporal dimension to a single time step.
SAMPLE_LENGTH = 16_384

# ── Model ──────────────────────────────────────────────────────────────────
NUM_CLASSES = 2   # Positive (1) vs Negative (0)

# ── Training ───────────────────────────────────────────────────────────────
BATCH_SIZE    = 32
NUM_EPOCHS    = 50
LEARNING_RATE = 0.01    # original paper: SGD lr=0.01
MOMENTUM      = 0.9
WEIGHT_DECAY  = 1e-7
PATIENCE      = 5       # epochs before LR is halved
NUM_WORKERS   = 4       # set 0 on Windows

GROUP_COLUMN  = 'Recording Team'   # column used for group-aware split

---
## 3 · Data loading

### Raw-waveform pipeline

Unlike the spectrogram notebook (where PNG images were pre-extracted),  
here we load `.wav` files at runtime and feed the raw samples directly  
to the model.  Every file is **padded or truncated** to exactly  
`SAMPLE_LENGTH` samples so all tensors in a batch have the same shape.

```
.wav file  →  load_audio()  →  pad_or_truncate()  →  normalise  →  tensor (1, 16384)
```

### Training augmentation

| Augmentation | Acoustic rationale |
|---|---|
| Random gain (×0.8–1.2) | Simulates microphone distance variation |
| Random time flip (50 %) | Acts as regularisation |
| Low-level Gaussian noise | Simulates background barn noise |
| ~~Pitch shift~~ | **Not used** — would alter fundamental frequency |
| ~~Time stretch~~ | **Not used** — would alter call-duration statistics |

In [ ]:
def load_audio(path, target_sr=SAMPLE_RATE):
    """Load a wav file to a 1-D float32 array at target_sr Hz."""
    if LIBROSA_AVAILABLE:
        y, _ = librosa.load(path, sr=target_sr, mono=True)
    else:
        y, sr = sf.read(path, dtype='float32', always_2d=False)
        if y.ndim > 1: y = y.mean(axis=1)
        if sr != target_sr:
            raise ValueError(f'Sample rate mismatch: {sr} vs {target_sr}. Install librosa.')
    return y.astype(np.float32)


def pad_or_truncate(y, length):
    """Force array to exactly *length* samples (truncate or zero-pad)."""
    if len(y) >= length:
        return y[:length]
    return np.pad(y, (0, length - len(y)), mode='constant')


class PigAudioDataset(Dataset):
    """
    Loads raw pig vocalisations from .wav files and returns
    (waveform_tensor, label) pairs.

    waveform_tensor shape: (1, SAMPLE_LENGTH)  — channel-first format
    label: 0 = Negative, 1 = Positive
    """

    def __init__(self, dataframe, audio_dir=AUDIO_DIR,
                 sample_length=SAMPLE_LENGTH, augment=False):
        self.df            = dataframe.reset_index(drop=True)
        self.audio_dir     = audio_dir
        self.sample_length = sample_length
        self.augment       = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = os.path.join(self.audio_dir, row['Audio Filename'])
        y    = load_audio(path)
        y    = pad_or_truncate(y, self.sample_length)

        if self.augment:
            y = self._augment(y)

        # Amplitude normalisation to [-1, 1]
        peak = np.abs(y).max()
        if peak > 0: y = y / peak

        waveform = torch.from_numpy(y).unsqueeze(0)        # (1, T)
        label    = torch.tensor(row['label'], dtype=torch.long)
        return waveform, label

    def _augment(self, y):
        y = y * np.random.uniform(0.8, 1.2)               # random gain
        if np.random.rand() < 0.5: y = y[::-1].copy()     # time flip
        noise = np.abs(y).mean() * 0.03
        y = y + (np.random.randn(len(y)) * noise).astype(np.float32)  # noise
        return y

---
## 4 · Train / Val / Test split  *(grouped by Recording Team)*

Pig IDs are encoded differently across the 6 recording teams:  
ETHZ → `pig15`, IASPA → `Pig8`, FBN → `VT13`, …  
Grouping by **Recording Team** prevents both within-pig and within-site  
data leakage — the same strategy used in the spectrogram notebook.

> 💡 If you standardise pig IDs into a single column, replace  
> `GROUP_COLUMN = 'Recording Team'` with that column name.

In [ ]:
df = pd.read_excel(ANNOTATIONS_FILE)
df['label'] = (df['Valence'] == 'Pos').astype(int)

groups = df[GROUP_COLUMN].values

# 80 / 10 / 10 group-aware split
gss_outer = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, temp_idx = next(gss_outer.split(df, groups=groups))

gss_inner = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_rel, test_rel = next(
    gss_inner.split(df.iloc[temp_idx], groups=groups[temp_idx])
)
val_idx  = temp_idx[val_rel]
test_idx = temp_idx[test_rel]

df_train = df.iloc[train_idx].reset_index(drop=True)
df_val   = df.iloc[val_idx].reset_index(drop=True)
df_test  = df.iloc[test_idx].reset_index(drop=True)

print(f'Train  {len(df_train):>5} samples | groups: {sorted(df_train[GROUP_COLUMN].unique())}')
print(f'Val    {len(df_val):>5} samples | groups: {sorted(df_val[GROUP_COLUMN].unique())}')
print(f'Test   {len(df_test):>5} samples | groups: {sorted(df_test[GROUP_COLUMN].unique())}')

# Build DataLoaders
train_ds = PigAudioDataset(df_train, augment=True)
val_ds   = PigAudioDataset(df_val,   augment=False)
test_ds  = PigAudioDataset(df_test,  augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

# Sanity-check one batch
w, l = next(iter(train_loader))
print(f'\nSample waveform batch shape: {tuple(w.shape)}')
print(f'Sample label batch shape   : {tuple(l.shape)}')

---
## 5 · SampleCNN model

### Architecture (power-of-2 variant)

```
Input        (B, 1, 16384)   ← mono raw waveform
Conv0 s=2    (B, 128, 8192)  ← learned filterbank (replaces STFT+Mel)
Block  1-2   (B, 128,  ?)    ← 128 filters, MaxPool(2) each
Block  3-12  (B, 256,  ?)    ← 256 filters, MaxPool(2) each
Block 13     (B, 512,  ?)    ← 512 filters, MaxPool(2)
Conv 1×1     (B, 512,  1)    ← channel mixing with collapsed time axis
Dropout(0.5)
Flatten      (B, 512)
Linear       (B,   2)        ← class logits
```

Total downsampling = 2 (stride) × 2^13 (13 pools) = **2^14 = 16 384**  
→ exactly collapses `SAMPLE_LENGTH = 16 384` to **1 time step** at the output.

In [ ]:
class SampleCNNBlock(nn.Module):
    """
    One convolutional block: Conv1d → BatchNorm → ReLU → MaxPool.

    BatchNorm is placed before the activation to stabilise training
    of this very deep network (14 conv layers total).
    bias=False in Conv1d because BatchNorm already has a learnable bias.
    """
    def __init__(self, in_ch, out_ch, kernel=2, pool=2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size=kernel, padding='same', bias=False),
            nn.BatchNorm1d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(pool),
        )

    def forward(self, x):
        return self.block(x)


class SampleCNN(nn.Module):
    """
    SampleCNN — power-of-2 variant, adapted for binary pig valence classification.

    Parameters
    ----------
    num_classes   : 2  (Pos / Neg)
    sample_length : 16384 = 2^14  (must match SAMPLE_LENGTH above)
    """
    def __init__(self, num_classes=2, sample_length=16_384):
        super().__init__()

        # Strided entry convolution — the 'learned filterbank'
        # kernel=2, stride=2 halves the time axis on the very first pass.
        # Compare this to STFT: STFT uses a fixed window; here the window
        # shape is optimised by backpropagation.
        self.conv0 = nn.Conv1d(1, 128, kernel_size=2, stride=2, bias=False)
        self.bn0   = nn.BatchNorm1d(128)
        self.relu0 = nn.ReLU(inplace=True)

        # 13 stacked blocks — each MaxPool(2) halves the time axis again.
        # Channel progression: 128 → 256 → 512 (more abstract features deeper)
        self.blocks = nn.Sequential(
            SampleCNNBlock(128, 128),   # block 1
            SampleCNNBlock(128, 128),   # block 2
            SampleCNNBlock(128, 256),   # block 3  ← channel expansion
            SampleCNNBlock(256, 256),   # block 4
            SampleCNNBlock(256, 256),   # block 5
            SampleCNNBlock(256, 256),   # block 6
            SampleCNNBlock(256, 256),   # block 7
            SampleCNNBlock(256, 256),   # block 8
            SampleCNNBlock(256, 256),   # block 9
            SampleCNNBlock(256, 256),   # block 10
            SampleCNNBlock(256, 256),   # block 11
            SampleCNNBlock(256, 256),   # block 12
            SampleCNNBlock(256, 512),   # block 13 ← channel expansion
        )

        # 1×1 convolution — at this point time=1, so this is equivalent to
        # a fully-connected layer over 512 channels, but more memory-efficient.
        self.conv1x1 = nn.Sequential(
            nn.Conv1d(512, 512, kernel_size=1, bias=False),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
        )

        self.dropout = nn.Dropout(p=0.5)
        self.flatten = nn.Flatten()
        self.fc      = nn.Linear(512, num_classes)

        # Kaiming (He) initialisation — designed for ReLU activations.
        # It keeps gradient variance stable across all 14 layers at init.
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)

    def forward(self, x):
        """x : (B, 1, sample_length)  →  logits : (B, num_classes)"""
        x = self.relu0(self.bn0(self.conv0(x)))   # learned filterbank
        x = self.blocks(x)                         # deep feature extraction
        x = self.conv1x1(x)                        # channel mixing
        x = self.dropout(x)
        x = self.flatten(x)                        # (B, 512)
        return self.fc(x)                          # (B, num_classes)


model = SampleCNN(num_classes=NUM_CLASSES, sample_length=SAMPLE_LENGTH).to(DEVICE)
total = sum(p.numel() for p in model.parameters())
print(f'SampleCNN — {total:,} parameters')

# Forward-pass shape trace
dummy = torch.randn(2, 1, SAMPLE_LENGTH).to(DEVICE)
with torch.no_grad():
    out = model(dummy)
print(f'Input shape  : {tuple(dummy.shape)}')
print(f'Output shape : {tuple(out.shape)}')

---
## 6 · Loss, optimiser and scheduler

| Choice | Reason |
|---|---|
| **SGD + Nesterov** | Matches original paper; often generalises better than Adam for deep CNNs |
| **Weighted CrossEntropy** | Dataset is ~2:1 Neg/Pos — up-weights the minority class |
| **ReduceLROnPlateau** | Automatically replicates the paper's manual LR schedule (0.01→0.002→…) |

In [ ]:
neg_count = (df_train['label'] == 0).sum()
pos_count = (df_train['label'] == 1).sum()
class_weights = torch.tensor([pos_count / neg_count, 1.0], dtype=torch.float).to(DEVICE)
print(f'Class weights → Neg: {class_weights[0]:.3f}  Pos: {class_weights[1]:.3f}')

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=LEARNING_RATE,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
    nesterov=True,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.2, patience=PATIENCE, verbose=True
)

---
## 7 · Training loop

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """One full pass over the training set with weight updates."""
    model.train()
    total_loss, all_labels, all_preds = 0.0, [], []
    for waveforms, labels in loader:
        waveforms, labels = waveforms.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(waveforms)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader.dataset), accuracy_score(all_labels, all_preds)


def evaluate(model, loader, criterion, device):
    """Evaluate model — no gradient updates."""
    model.eval()
    total_loss, all_labels, all_preds, all_probs = 0.0, [], [], []
    with torch.no_grad():
        for waveforms, labels in loader:
            waveforms, labels = waveforms.to(device), labels.to(device)
            logits = model(waveforms)
            loss   = criterion(logits, labels)
            probs  = torch.softmax(logits, dim=1)[:, 1]
            total_loss += loss.item() * len(labels)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    avg_loss = total_loss / len(loader.dataset)
    return avg_loss, accuracy_score(all_labels, all_preds), np.array(all_labels), np.array(all_probs)

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_loss = float('inf')

print(f"{'Epoch':>5}  {'Train Loss':>10}  {'Train Acc':>9}  {'Val Loss':>8}  {'Val Acc':>7}  {'LR':>8}")
print('─' * 65)

for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    vl_loss, vl_acc, _, _ = evaluate(model, val_loader, criterion, DEVICE)
    scheduler.step(vl_loss)

    history['train_loss'].append(tr_loss); history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc);  history['val_acc'].append(vl_acc)

    lr = optimizer.param_groups[0]['lr']
    flag = ''
    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        flag = '  ← saved'

    print(f"{epoch:>5}  {tr_loss:>10.4f}  {tr_acc:>9.4f}  {vl_loss:>8.4f}  {vl_acc:>7.4f}  {lr:>8.6f}{flag}")

print('\nTraining complete.')

---
## 8 · Learning curves

In [ ]:
ep = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(ep, history['train_loss'], label='Train')
axes[0].plot(ep, history['val_loss'],   label='Val')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss curves'); axes[0].legend()

axes[1].plot(ep, history['train_acc'], label='Train')
axes[1].plot(ep, history['val_acc'],   label='Val')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy curves'); axes[1].legend()

plt.tight_layout()
plt.show()

---
## 9 · Test evaluation

We load the **best checkpoint** (lowest val loss) before evaluating —  
not the weights from the final epoch, which may be over-fitted.

**Why AUC-ROC?**  
The dataset is imbalanced (~2:1 Neg/Pos). AUC-ROC measures how well the  
model *ranks* positive examples above negative ones across all thresholds,  
making it more informative than accuracy alone.

In [ ]:
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))

test_loss, test_acc, test_labels, test_probs = evaluate(model, test_loader, criterion, DEVICE)
test_preds = (test_probs >= 0.5).astype(int)
auc = roc_auc_score(test_labels, test_probs)
ap  = average_precision_score(test_labels, test_probs)

print('── Test-set results ──────────────────────────────────────────')
print(f'  Loss               : {test_loss:.4f}')
print(f'  Accuracy           : {test_acc:.4f}')
print(f'  AUC-ROC            : {auc:.4f}   (1.0=perfect, 0.5=random)')
print(f'  Average Precision  : {ap:.4f}')
print()
print(classification_report(test_labels, test_preds, target_names=['Neg', 'Pos']))

In [ ]:
cm = confusion_matrix(test_labels, test_preds)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Neg', 'Pos'], yticklabels=['Neg', 'Pos'], ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'Confusion matrix — AUC = {auc:.3f}')
plt.tight_layout()
plt.show()

---
## 10 · What to try next

| Idea | How |
|---|---|
| **Power-of-3 variant** | Change `SampleCNNBlock(kernel=3, pool=3)` and set `SAMPLE_LENGTH = 3^9 = 19683` |
| **Segment averaging** | Split each long call into N chunks, average model predictions (as in the original paper) |
| **Compare with spectrogram CNN** | The other notebook uses ResNet-18 on Mel spectrograms — compare AUC-ROC scores |
| **Per-pig split** | Add a standardised `pig_id` column and swap `GROUP_COLUMN` for finer leakage control |
| **Adam optimiser** | Replace SGD with `torch.optim.Adam(lr=1e-4)` and observe convergence speed |
| **Metadata fusion** | Concatenate age, sex, call-type embeddings to the 512-dim feature vector before the head |